# 04 Judge
Attach: repo zip dataset, 03_generate output. Configure the notebook as a Kaggle utility benchmark so model-proxy access is enabled.

In [ ]:
import glob, os, shutil, zipfile
matches = glob.glob("/kaggle/input/**/submission_adapter.py", recursive=True)
if not matches:
    for archive in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        zipfile.ZipFile(archive).extractall("/tmp/unzipped")
    matches = glob.glob("/tmp/unzipped/**/submission_adapter.py", recursive=True)
assert matches, "attach the repo zip as a Kaggle dataset"
REPO = "/tmp/repo"
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(os.path.dirname(matches[0]), REPO)
os.chdir(REPO)
os.environ["PYTHONPATH"] = REPO
OUT = "/kaggle/working/outputs"
os.makedirs(OUT, exist_ok=True)
print(os.listdir(REPO))

In [ ]:
!python -m pip install -q -e . --no-deps
import kaggle_benchmarks as kbench
print(kbench.__name__)

In [ ]:
def find_file(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, name
    return hits[0]

PAIRS_ALPACA = find_file("dpo_vs_sft_alpaca_pairs.jsonl")
PAIRS_HH = find_file("dpo_vs_sft_hh_pairs.jsonl")
PAIRS_PERSONA = find_file("lora_vs_base_persona_pairs.jsonl")
JUDGE = "google/gemini-3.6-flash"
os.makedirs(f"{OUT}/eval", exist_ok=True)

## Smoke test (5 pairs)

In [ ]:
!python scripts/judge_pairwise_kaggle.py --pairs_path {PAIRS_ALPACA} --output_path /tmp/judge_smoke.jsonl --judge_model {JUDGE} --limit 5 --batch_size 5 --sleep_seconds 5 --max_retries 3

## Full judging

In [ ]:
!python scripts/judge_pairwise_kaggle.py --pairs_path {PAIRS_ALPACA} --output_path {OUT}/eval/dpo_vs_sft_alpaca_judged.jsonl --judge_model {JUDGE} --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments
!python scripts/judge_pairwise_kaggle.py --pairs_path {PAIRS_HH} --output_path {OUT}/eval/dpo_vs_sft_hh_judged.jsonl --judge_model {JUDGE} --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments
!python scripts/judge_pairwise_kaggle.py --pairs_path {PAIRS_PERSONA} --output_path {OUT}/eval/lora_vs_base_persona_judged.jsonl --judge_model {JUDGE} --subject_name lora_dpo --baseline_name smollm2_instruct --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments